# Paper PPO Reproduction

Notebook version of `src/ppo/paper_ppo_reproduction.py`.

This keeps the paper-style environment but saves everything under `artifacts/ppo/paper_ppo_reproduction`.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "ppo"
RESULTS_ROOT = PROJECT_ROOT / "artifacts" / "ppo" / "paper_ppo_reproduction"
SCRIPT_DIR = PROJECT_ROOT / "src" / "deep_learning" / "Elurant_PPO"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import paper_ppo_reproduction as paper

paper.MODELS_DIR = RESULTS_ROOT / "models"
paper.TB_DIR = RESULTS_ROOT / "tensorboard"
paper.LOGS_DIR = RESULTS_ROOT / "logs"
paper.MODELS_DIR.mkdir(parents=True, exist_ok=True)
paper.TB_DIR.mkdir(parents=True, exist_ok=True)
paper.LOGS_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook dir:", NOTEBOOK_DIR)
print("Script dir:", SCRIPT_DIR)
print("Results dir:", RESULTS_ROOT)


## Config

In [ ]:
cfg = {
    'timesteps': 10000,
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'batch_size': 256,
    'n_steps': 512,
    'n_epochs': 10,
    'ent_coef': 0.01,
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
    'eval_episodes': 5,
    'n_envs': 3,
    'eval_freq': 3000,
    'seed': 42,
    'device': 'auto',
    'progress_every': 3000,
    'run_adaptability_tests': False,
}

print(json.dumps(cfg, indent=2))


## Train And Evaluate

In [ ]:
env = paper.make_vec_env(paper.make_paper_env, n_envs=cfg['n_envs'], seed=cfg['seed'])

model = paper.PPO(
    policy='MlpPolicy',
    env=env,
    learning_rate=cfg['learning_rate'],
    n_steps=cfg['n_steps'],
    batch_size=cfg['batch_size'],
    n_epochs=cfg['n_epochs'],
    gamma=cfg['gamma'],
    gae_lambda=cfg['gae_lambda'],
    clip_range=cfg['clip_range'],
    ent_coef=cfg['ent_coef'],
    vf_coef=cfg['vf_coef'],
    max_grad_norm=cfg['max_grad_norm'],
    verbose=1,
    tensorboard_log=str(paper.TB_DIR),
    seed=cfg['seed'],
    device=cfg['device'],
    policy_kwargs={'net_arch': {'pi': [256, 256], 'vf': [256, 256]}},
)

eval_env = paper.Monitor(paper.make_paper_env())
best_model_dir = paper.MODELS_DIR / 'best_model'
best_model_dir.mkdir(parents=True, exist_ok=True)
eval_callback = paper.EvalCallback(
    eval_env,
    best_model_save_path=str(best_model_dir),
    log_path=str(paper.LOGS_DIR),
    eval_freq=max(1, cfg['eval_freq'] // max(1, cfg['n_envs'])),
    deterministic=True,
    render=False,
)
progress_callback = paper.TimestepProgressCallback(total_timesteps=cfg['timesteps'], every_n_steps=cfg['progress_every'])

start = time.time()
model.learn(total_timesteps=cfg['timesteps'], callback=[progress_callback, eval_callback], tb_log_name='paper_ppo_reproduction', progress_bar=False)
elapsed = time.time() - start

model_path = paper.MODELS_DIR / 'paper_ppo_reproduction'
model.save(str(model_path))
best_model_path = best_model_dir / 'best_model.zip'
eval_model = paper.PPO.load(str(best_model_path if best_model_path.exists() else model_path.with_suffix('.zip')))
evaluation = paper.evaluate_model(eval_model, episodes=cfg['eval_episodes'], base_seed=cfg['seed'] + 10_000)
evaluation['elapsed_seconds'] = float(elapsed)
evaluation['model_path'] = str(model_path.with_suffix('.zip'))
evaluation['best_model_path'] = str(best_model_path) if best_model_path.exists() else None
(paper.MODELS_DIR / 'evaluation.json').write_text(json.dumps(evaluation, indent=2), encoding='utf-8')
print(json.dumps(evaluation, indent=2))

eval_env.close()
env.close()


## Evaluation Helpers


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display


def learned_policy_model_path() -> Path:
    best_path = paper.MODELS_DIR / 'best_model' / 'best_model.zip'
    final_path = paper.MODELS_DIR / 'paper_ppo_reproduction.zip'
    if best_path.exists():
        return best_path
    if final_path.exists():
        return final_path
    raise RuntimeError('Run the training cell once so a learned PPO model exists.')


def load_learned_policy_model():
    model_path = learned_policy_model_path()
    loaded_model = paper.PPO.load(str(model_path), device=cfg.get('device', 'auto'))
    print(f'Loaded learned policy from: {model_path}')
    return loaded_model, model_path


def compute_same_lane_ttc(env, ttc_cap: float = 10.0) -> float:
    vehicle = getattr(env.unwrapped, 'vehicle', None)
    road = getattr(env.unwrapped, 'road', None)
    if vehicle is None or road is None:
        return float(ttc_cap)

    lane_index = getattr(vehicle, 'lane_index', None)
    if lane_index is None:
        return float(ttc_cap)

    front_vehicle, _ = road.neighbour_vehicles(vehicle, lane_index)
    if front_vehicle is None:
        return float(ttc_cap)

    lane = road.network.get_lane(lane_index)
    ego_s, _ = lane.local_coordinates(vehicle.position)
    front_s, _ = lane.local_coordinates(front_vehicle.position)
    clearance = max(
        0.0,
        float(front_s - ego_s)
        - 0.5 * float(getattr(vehicle, 'LENGTH', 0.0) + getattr(front_vehicle, 'LENGTH', 0.0)),
    )
    ego_speed = float(vehicle.speed * np.cos(getattr(vehicle, 'heading', 0.0)))
    front_speed = float(front_vehicle.speed * np.cos(getattr(front_vehicle, 'heading', 0.0)))
    closing_speed = ego_speed - front_speed
    if closing_speed <= 1e-6:
        return float(ttc_cap)

    ttc = 0.0 if clearance <= 0.0 else clearance / closing_speed
    return float(np.clip(ttc, 0.0, ttc_cap))


def collect_paper_policy_eval(
    model,
    *,
    episodes: int,
    seed: int,
    render_mode: str | None = None,
    config_overrides: dict | None = None,
    ttc_cap: float = 10.0,
    max_steps: int | None = None,
    render_sleep: float = 0.0,
    progress_every: int = 100,
) -> list[dict[str, float | bool | int]]:
    rows = []
    for episode_idx in range(int(episodes)):
        env = paper.Monitor(paper.make_paper_env(render_mode=render_mode, config_overrides=config_overrides))
        try:
            obs, _ = env.reset(seed=int(seed) + episode_idx)
            if render_mode == 'human':
                env.render()
            terminated = False
            truncated = False
            total_reward = 0.0
            step_count = 0
            speed_trace = []
            ttc_trace = []
            final_info = {}

            while not (terminated or truncated):
                speed_trace.append(float(getattr(env.unwrapped.vehicle, 'speed', 0.0)))
                ttc_trace.append(compute_same_lane_ttc(env, ttc_cap=ttc_cap))

                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += float(reward)
                step_count += 1
                final_info = dict(info)

                if render_mode == 'human':
                    env.render()
                    if render_sleep > 0.0:
                        time.sleep(float(render_sleep))

                if max_steps is not None and step_count >= int(max_steps):
                    truncated = True
                    break

            metrics = dict(final_info.get('episode_metrics', {}))
            row = {
                'episode': int(episode_idx + 1),
                'collision': bool(metrics.get('collision', getattr(env.unwrapped.vehicle, 'crashed', False))),
                'avg_speed': float(np.mean(speed_trace)) if speed_trace else 0.0,
                'overtakes': int(metrics.get('overtaken_count', final_info.get('overtaken_count', 0))),
                'avg_ttc': float(np.mean(ttc_trace)) if ttc_trace else float(ttc_cap),
                'min_ttc': float(np.min(ttc_trace)) if ttc_trace else float(ttc_cap),
                'reward': float(total_reward),
                'steps': int(step_count),
                'success': bool(metrics.get('success', final_info.get('success', 0.0))),
                'offroad': bool(metrics.get('offroad', final_info.get('offroad', 0.0))),
            }
            rows.append(row)

            if progress_every and ((episode_idx + 1) % int(progress_every) == 0 or render_mode == 'human'):
                print(
                    f"[eval] episode={episode_idx + 1}/{episodes} "
                    f"reward={row['reward']:.2f} avg_speed={row['avg_speed']:.2f} "
                    f"min_ttc={row['min_ttc']:.2f} collision={row['collision']}"
                )
        finally:
            env.close()
    return rows


def metric_columns(eval_df: pd.DataFrame) -> list[str]:
    columns = ['episode', 'collision', 'avg_speed', 'overtakes', 'avg_ttc', 'min_ttc', 'reward', 'steps']
    return [column for column in columns if column in eval_df.columns]


def build_metric_summary(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column, label, scale in [
        ('avg_speed', 'Average Speed (m/s)', 1.0),
        ('collision', 'Collision Rate (%)', 100.0),
        ('overtakes', 'Overtakes', 1.0),
        ('avg_ttc', 'Average TTC (s)', 1.0),
        ('min_ttc', 'Minimum TTC (s)', 1.0),
        ('reward', 'Reward', 1.0),
    ]:
        values = pd.to_numeric(eval_df[column], errors='coerce').fillna(0.0).astype(float) * scale
        std = float(values.std(ddof=0))
        standard_error = float(std / np.sqrt(max(len(values), 1)))
        row = {'metric': label, 'mean': float(values.mean()), 'std': std, 'standard_error': standard_error}
        if column == 'collision':
            row['collisions'] = int(pd.to_numeric(eval_df[column], errors='coerce').fillna(0).astype(bool).sum())
            row['episodes'] = int(len(values))
        rows.append(row)
    return pd.DataFrame(rows)


def plot_evaluation_metrics(summaries: list[dict[str, float | bool | int]], save_path: Path) -> None:
    if not summaries:
        return

    episodes = np.arange(1, len(summaries) + 1)
    avg_speed = np.array([float(item['avg_speed']) for item in summaries], dtype=float)
    overtakes = np.array([float(item['overtakes']) for item in summaries], dtype=float)
    avg_ttc = np.array([float(item['avg_ttc']) for item in summaries], dtype=float)
    min_ttc = np.array([float(item['min_ttc']) for item in summaries], dtype=float)
    collisions = np.array([float(bool(item['collision'])) for item in summaries], dtype=float)
    running_collision_rate = 100.0 * np.cumsum(collisions) / np.arange(1, len(collisions) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].plot(episodes, avg_speed, marker='o', color='tab:green')
    axes[0, 0].set_title('Average Speed')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('m/s')
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(episodes, running_collision_rate, marker='o', color='crimson')
    axes[0, 1].set_title('Running Collision Rate')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('%')
    axes[0, 1].set_ylim(0.0, 100.0)
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].bar(episodes, overtakes, color='tab:orange')
    axes[1, 0].set_title('Overtakes')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].grid(axis='y', alpha=0.3)

    axes[1, 1].plot(episodes, min_ttc, marker='o', label='Min TTC', color='tab:blue')
    axes[1, 1].plot(episodes, avg_ttc, marker='o', label='Avg TTC', color='tab:purple')
    axes[1, 1].set_title('Time To Collision')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Seconds')
    axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend()

    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def plot_metric_summary(eval_df: pd.DataFrame, save_path: Path, title: str) -> pd.DataFrame:
    specs = [
        ('avg_speed', 'Average Speed (m/s)', 'tab:green', 1.0),
        ('collision', 'Collision Rate (%)', 'crimson', 100.0),
        ('overtakes', 'Overtakes', 'tab:orange', 1.0),
        ('avg_ttc', 'Average TTC (s)', 'tab:purple', 1.0),
        ('min_ttc', 'Minimum TTC (s)', 'tab:blue', 1.0),
        ('reward', 'Reward', 'tab:gray', 1.0),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    episode_label = f'{len(eval_df)} eps'
    for ax, (column, label, color, scale) in zip(axes.flat, specs):
        values = pd.to_numeric(eval_df[column], errors='coerce').fillna(0.0).astype(float) * scale
        mean = float(values.mean())
        std = float(values.std(ddof=0))
        standard_error = float(std / np.sqrt(max(len(values), 1)))
        error = standard_error if column == 'collision' else std
        error_label = 'se' if column == 'collision' else 'std'
        ax.bar([0], [mean], yerr=[error], capsize=8, color=color, alpha=0.85)
        ax.scatter(np.zeros(len(values)), values, color=color, alpha=0.08, s=8)
        ax.set_xticks([0], [episode_label])
        ax.set_title(f'{label}\nmean={mean:.2f}, {error_label}={error:.2f}')
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle(title)
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return build_metric_summary(eval_df)


## Visible Learned-Policy Evaluation


In [ ]:
visible_eval_episodes = 5
visible_eval_seed = cfg['seed'] + 20_000
visible_eval_max_steps = 300

visible_model, visible_model_path = load_learned_policy_model()
visible_eval_rows = collect_paper_policy_eval(
    visible_model,
    episodes=visible_eval_episodes,
    seed=visible_eval_seed,
    render_mode='human',
    max_steps=visible_eval_max_steps,
    render_sleep=0.05,
    progress_every=1,
)
visible_eval_df = pd.DataFrame(visible_eval_rows)
print(f'Visible evaluation model: {visible_model_path}')
display(visible_eval_df[metric_columns(visible_eval_df)].round(3))


## Learned-Policy Evaluation on 1,000 Episodes


In [ ]:
learned_policy_eval_episodes = 1000
learned_policy_eval_seed = cfg['seed'] + 30_000
learned_policy_eval_dir = RESULTS_ROOT / f'learned_policy_eval_{learned_policy_eval_episodes}_episodes'
learned_policy_eval_dir.mkdir(parents=True, exist_ok=True)

learned_policy_model, learned_policy_model_path = load_learned_policy_model()
learned_policy_eval_rows = collect_paper_policy_eval(
    learned_policy_model,
    episodes=learned_policy_eval_episodes,
    seed=learned_policy_eval_seed,
    render_mode=None,
    progress_every=100,
)
learned_policy_eval_df = pd.DataFrame(learned_policy_eval_rows)

metrics_path = learned_policy_eval_dir / 'evaluation_metrics.json'
detail_plot_path = learned_policy_eval_dir / 'evaluation_metrics.png'
summary_plot_path = learned_policy_eval_dir / 'evaluation_summary.png'
summary_path = learned_policy_eval_dir / 'evaluation_summary.json'

metrics_path.write_text(learned_policy_eval_df.to_json(orient='records', indent=2), encoding='utf-8')
plot_evaluation_metrics(learned_policy_eval_rows, detail_plot_path)
metric_summary_df = plot_metric_summary(
    learned_policy_eval_df,
    summary_plot_path,
    f'Paper PPO Learned-Policy Evaluation ({learned_policy_eval_episodes} episodes)',
)

learned_policy_eval_summary = {
    'episodes': int(learned_policy_eval_episodes),
    'seed': int(learned_policy_eval_seed),
    'model_path': str(learned_policy_model_path),
    'evaluation_metrics_path': str(metrics_path),
    'detailed_plot_path': str(detail_plot_path),
    'summary_plot_path': str(summary_plot_path),
    'metric_summary': metric_summary_df.to_dict(orient='records'),
}
summary_path.write_text(json.dumps(learned_policy_eval_summary, indent=2), encoding='utf-8')

print(json.dumps(learned_policy_eval_summary, indent=2))
display(Image(filename=detail_plot_path))
display(Image(filename=summary_plot_path))
display(metric_summary_df.round(3))
display(learned_policy_eval_df[metric_columns(learned_policy_eval_df)].head(10).round(3))
